
# Differential Expression → bnlearn Constraints

**Input:** the four `sound-life_*_deseq2-results_2025-02-07.csv` files — gene-level
DESeq2 results per AIFI_L3 cell type (`AIFI_L3, fg, bg, gene, log2fc, padj, pvalue, stat`),
one file per contrast (age group, biological sex, CMV status, flu vaccine).

**These are summary statistics, not observations.** They are *never* rows or columns
of the BN matrix. This notebook turns them into **edge constraints** for the R/bnlearn
pipeline, in two clearly separated parts:

- **Part A — structural blacklist (primary deliverable, fed to `hc()`).**
  Pure causal/CLG logic, *no DE used*: nothing causes a person's age/sex/CMV, and the
  three discrete roots are mutually independent. Outputs `bn_node_tiers.csv`
  (source of truth) + a materialized `bn_constraints.csv`.

- **Part B — back-pocket biology-first whitelist (uses DE; NOT auto-loaded to bnlearn).**
  A small set of `root → pathway` edges pre-registered from *independent* primary
  literature, then localized to specific cell-type pseudobulk nodes using GSEA on the
  DE statistics. Kept as candidates in `de_whitelist_candidates.csv` for the team to
  review edge-by-edge — deliberately **not** wired into the model.

**Design rationale (flat prior / anti-circularity).** The project imposes *no prior on
which edges exist or their direction*. Blacklists encode only impossibilities (safe).
Deriving edge *existence* from the same scRNA-seq the nodes are built from would leak the
data's association structure — so DE-driven whitelists are biology-first (literature
asserts the edge; DE only localizes it) and are held out of the default model.

> **Not a `*_baseline_wide.csv` block:** this modality produces constraint tables, so the
> shared export contract (`io.enforce_export_contract` / `write_processed`) and
> `feature_manifest.csv` do **not** apply. Stage 2 (baseline filtering) is also N/A —
> DESeq2 already collapsed samples into a summary statistic.

## Setup & configuration

In [1]:
import csv
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.transforms import to_snake_case

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Discrete root nodes (the three exogenous parents) -----------------------
# Names MUST match Person A's final clinical_baseline_wide.csv columns. Verified
# against data/raw/sound_life_labs_metadata.csv header (2026-07-08). The CMV name
# is a PLACEHOLDER (raw column) — reconcile with Person A before the R pipeline runs.
ROOT_NODES = {
    "age_group": "subject.ageGroup",
    "sex":       "subject.biologicalSex",
    "cmv":       "cmv.igg_serology_interpretation",   # placeholder
}
ROOT_COLS = list(ROOT_NODES.values())

# --- DE input files (constraints, NOT nodes) ---------------------------------
DE_FILES = {
    "age_group":   RAW / "sound-life_age-group_deseq2-results_2025-02-07 (1).csv",
    "sex":         RAW / "sound-life_biological-sex_deseq2-results_2025-02-07 (1).csv",
    "cmv":         RAW / "sound-life_cmv-status_deseq2-results_2025-02-07 (1).csv",
    "flu_vaccine": RAW / "sound-life_flu-vaccine_deseq2-results_2025-02-07 (1).csv",  # PARKED
}

# --- GSEA gate thresholds (conventional; NOT tuned on this dataset) ----------
FDR_THRESHOLD = 0.05     # enrichment FDR q-value gate
GSEA_MIN_SIZE = 10
GSEA_MAX_SIZE = 500
GSEA_PERM     = 1000
GSEA_SEED     = 0

# --- Hallmark pathway keys: reused VERBATIM from Person B's pseudobulk build --
# gp.get_library("MSigDB_Hallmark_2020", organism="Human"); human-readable keys.
PATHWAY_SETS = {
    "ifn_gamma":     "Interferon Gamma Response",
    "ifn_alpha":     "Interferon Alpha Response",
    "inflammatory":  "Inflammatory Response",
    "tnfa_nfkb":     "TNF-alpha Signaling via NF-kB",
    "il2_stat5":     "IL-2/STAT5 Signaling",
    "il6_jak_stat3": "IL-6/JAK/STAT3 Signaling",
}


### Pre-registered biology-first hypotheses (Step A)

Each `root → pathway` edge is asserted from **independent primary literature** — not this
project's data and not Person B's manifest (which shares this project's framing). DE is
used only to *localize* each edge to cell types (Step B, Part B). All PMIDs verified
against PubMed on 2026-07-08.

| Edge | Direction (biology) | DE contrast (fg vs bg) | Expected NES sign | Citation |
|------|--------------------|------------------------|-------------------|----------|
| age → inflammatory | Older ↑ | Older vs Younger | **+** | Franceschi 2000 (PMID 10911963); Ferrucci & Fabbri 2018 (30065258) |
| age → il6_jak_stat3 | Older ↑ | Older vs Younger | **+** | Ferrucci & Fabbri 2018 (30065258) |
| age → tnfa_nfkb | Older ↑ | Older vs Younger | **+** | Franceschi 2000 (10911963) |
| CMV⁺ → ifn_gamma | CMV⁺ ↑ | Positive vs Negative | **+** | Sylwester 2005 (16147978) |
| sex → ifn_alpha | Female ↑ | **Male vs Female** | **−** | Griesbeck 2015 (26519527); Klein & Flanagan 2016 (27546235) |

The **sex** row is the subtle one: the DE contrast is Male-vs-Female (`fg=Male`), so a
female-elevated pathway must show a *negative* NES. `CMV → tnfa_nfkb` was deliberately
**dropped** — Sylwester confirms IFN-γ but not TNF, so we do not assert an under-sourced edge.

In [2]:
# expected_nes_sign encodes the DE contrast direction (fg vs bg per file).
HYPOTHESES = [
    dict(root="age_group", pathway="inflammatory",  expected_nes_sign=+1,
         citation="Franceschi 2000 PMID:10911963; Ferrucci & Fabbri 2018 PMID:30065258"),
    dict(root="age_group", pathway="il6_jak_stat3", expected_nes_sign=+1,
         citation="Ferrucci & Fabbri 2018 PMID:30065258"),
    dict(root="age_group", pathway="tnfa_nfkb",     expected_nes_sign=+1,
         citation="Franceschi 2000 PMID:10911963"),
    dict(root="cmv",       pathway="ifn_gamma",     expected_nes_sign=+1,
         citation="Sylwester 2005 PMID:16147978"),
    dict(root="sex",       pathway="ifn_alpha",     expected_nes_sign=-1,
         citation="Griesbeck 2015 PMID:26519527; Klein & Flanagan 2016 PMID:27546235"),
]

## Stage 1 — Ingestion & standardization

In [3]:
# One row = one gene x cell type x contrast. Constraints, not observations.
de = {}
for name, path in DE_FILES.items():
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    de[name] = df
    print(f"{name:12s} {df.shape[0]:>7,} rows | "
          f"{df['AIFI_L3'].nunique():>3} L3 types | "
          f"{df['gene'].nunique():>6,} genes | "
          f"contrast: {df['fg'].iloc[0]} vs {df['bg'].iloc[0]}")

age_group    373,942 rows |  71 L3 types | 11,454 genes | contrast: Older Adult vs Younger Adult


sex          373,942 rows |  71 L3 types | 11,454 genes | contrast: Male vs Female


cmv          373,942 rows |  71 L3 types | 11,454 genes | contrast: Positive vs Negative


flu_vaccine  376,678 rows |  71 L3 types | 11,508 genes | contrast: Flu Year 1 Day 7 vs Flu Year 1 Day 0



**Park the flu-vaccine contrast.** Flu Year 1 Day 7 vs Day 0 is a *temporal*
(post-vaccination) contrast. Flu response is a held-out validation axis with **no baseline
training node**, so it cannot constrain any edge in the baseline BN. Loaded above for
transparency, excluded from all constraint generation below.

In [4]:
ACTIVE_CONTRASTS = ["age_group", "sex", "cmv"]
print("Active contrasts:", ACTIVE_CONTRASTS)
print("Parked (loaded, not used):", [k for k in de if k not in ACTIVE_CONTRASTS])

Active contrasts: ['age_group', 'sex', 'cmv']
Parked (loaded, not used): ['flu_vaccine']



## Part A — Structural blacklist (primary deliverable)

**Rule 1** — no edges *into* the discrete roots. Nothing in the cohort causes a person's
age group, sex, or CMV serostatus; this is also required by the CLG assumption (discrete
nodes cannot have continuous parents).
**Rule 2** — the three roots are mutually independent (no edges among them).

Both come from causal logic + CLG, **not** from DE. Nothing *outbound* from the roots is
constrained — every `root → X` edge is left entirely to the learner. Iterating "every edge
into each root, from every other node (including the other roots)" covers Rules 1 and 2 at once.

In [5]:
# Discover the molecular node universe from the completed wide blocks. Read the
# header via csv (not pandas) so duplicate column names are not mangled, then keep
# exact-unique feature columns (drop identifiers and QC flag columns).
MOLECULAR_FILES = [
    "celltype_freq_baseline_wide.csv",
    "olink_baseline_wide.csv",
    "pseudobulk_baseline_wide.csv",
    "wholeblood_baseline_wide.csv",
]
ID_COLS = {"subject.subjectGuid", "sample.sampleKitGuid"}

def is_node(col):
    return col not in ID_COLS and "flag" not in col.lower()

molecular_nodes = []
for f in MOLECULAR_FILES:
    with open(PROCESSED / f, newline="") as fh:
        header = next(csv.reader(fh))
    molecular_nodes += [c for c in header if is_node(c)]
molecular_nodes = list(dict.fromkeys(molecular_nodes))   # exact de-dup, stable order
print(f"molecular nodes discovered: {len(molecular_nodes)}")
print(f"  (roots added separately: {ROOT_COLS})")

molecular nodes discovered: 578
  (roots added separately: ['subject.ageGroup', 'subject.biologicalSex', 'cmv.igg_serology_interpretation'])


In [6]:
def build_blacklist(nodes, roots):
    # Every edge INTO a root is forbidden (covers Rule 1 and Rule 2).
    node_set = list(dict.fromkeys(list(nodes) + list(roots)))
    rows = [{"from": n, "to": r, "constraint": "blacklist"}
            for r in roots for n in node_set if n != r]
    return pd.DataFrame(rows)

blacklist = build_blacklist(molecular_nodes, ROOT_COLS)
n_other = len(dict.fromkeys(molecular_nodes + ROOT_COLS)) - 1
print(f"blacklist edges: {len(blacklist)}  (= {len(ROOT_COLS)} roots x {n_other} other nodes)")
blacklist.head()

blacklist edges: 1740  (= 3 roots x 580 other nodes)


,from,to,constraint
0,freq.L1_b_cell,subject.ageGroup,blacklist
1,freq.L1_dc,subject.ageGroup,blacklist
2,freq.L1_erythrocyte,subject.ageGroup,blacklist
3,freq.L1_ilc,subject.ageGroup,blacklist
4,freq.L1_monocyte,subject.ageGroup,blacklist



### Encoding (source of truth + materialized form)

`bn_node_tiers.csv` is the **authoritative, regenerable** artifact: tier 0 = the three
roots, tier 1 = every other node (any node not listed defaults to tier 1). The R side
expands it with `bnlearn::tiers2blacklist()`. `bn_constraints.csv` is the materialized
`from,to,constraint` form for direct use today (every row `blacklist`).

> ⚠️ **Regenerate after the join.** This is materialized against the *current* molecular
> blocks + the three roots. Person A's clinical nodes are not yet in `data/processed/`, so
> once `bn_ready_baseline.csv` exists, rerun `build_blacklist(final_nodes, ROOT_COLS)` (or
> `tiers2blacklist` in R) against the final column list — `hc()` errors if a blacklist edge
> names a node absent from the data.

In [7]:
tiers = pd.DataFrame(
    [{"node": r, "tier": 0} for r in ROOT_COLS]
    + [{"node": n, "tier": 1} for n in molecular_nodes]
)
tiers.to_csv(PROCESSED / "bn_node_tiers.csv", index=False)
blacklist.to_csv(PROCESSED / "bn_constraints.csv", index=False)
print("wrote bn_node_tiers.csv :", tiers.shape, "| tier counts:",
      dict(tiers["tier"].value_counts().sort_index()))
print("wrote bn_constraints.csv:", blacklist.shape)

wrote bn_node_tiers.csv : (581, 2) | tier counts: {0: np.int64(3), 1: np.int64(578)}
wrote bn_constraints.csv: (1740, 3)



## Part B — Back-pocket biology-first whitelist (uses DE)

**Step B: localize each pre-registered `root → pathway` edge to cell types.** For each
active contrast and each AIFI_L3 cell type, run **GSEA prerank** on the signed Wald
statistic (`stat`) against the Hallmark gene set — the *same* `MSigDB_Hallmark_2020` sets
Person B used to build the `pb.*` scores. A candidate edge `root → pb.<celltype>_<pathway>`
is emitted only where enrichment **FDR < 0.05** *and* the **NES sign matches** the
literature-expected direction. GSEA uses the full signed ranking (no arbitrary
significant-gene cutoff); thresholds are textbook conventions, not tuned on this data.

Output is **candidates only** — `de_whitelist_candidates.csv` with full provenance. It is
**not** loaded into `hc()`; the team decides edge-by-edge whether to promote any.

In [8]:
import gseapy as gp

# Reuse the EXACT library Person B used, so the gene -> pb-node bridge is consistent.
hallmark = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")
gene_sets = {abbr: hallmark[key] for abbr, key in PATHWAY_SETS.items()}
for abbr, key in PATHWAY_SETS.items():
    print(f"{abbr:14s} <- '{key}': {len(gene_sets[abbr])} genes")

ifn_gamma      <- 'Interferon Gamma Response': 200 genes
ifn_alpha      <- 'Interferon Alpha Response': 97 genes
inflammatory   <- 'Inflammatory Response': 200 genes
tnfa_nfkb      <- 'TNF-alpha Signaling via NF-kB': 200 genes
il2_stat5      <- 'IL-2/STAT5 Signaling': 199 genes
il6_jak_stat3  <- 'IL-6/JAK/STAT3 Signaling': 87 genes


In [9]:
# Node-anchored L3 -> pb mapping. DE L3 names collapse under to_snake_case
# (e.g. GZMB+/- variants), so emit a candidate ONLY when the stem matches a real
# pb node column; the original L3 string is preserved in provenance so any
# many-to-one collisions stay visible for review.
with open(PROCESSED / "pseudobulk_baseline_wide.csv", newline="") as fh:
    pb_header = next(csv.reader(fh))
pb_nodes = {c for c in pb_header if c.startswith("pb.") and "flag" not in c}

def pb_node_for(l3_name, pathway):
    node = f"pb.{to_snake_case(l3_name)}_{pathway}"
    return node if node in pb_nodes else None

# quick coverage check
mapped = sum(pb_node_for(l3, "inflammatory") is not None
             for l3 in de["age_group"]["AIFI_L3"].unique())
print(f"L3 types mapping to a pb node: {mapped} / {de['age_group']['AIFI_L3'].nunique()}")

L3 types mapping to a pb node: 71 / 71


In [10]:
def prerank_celltype(df_ct, sets):
    # Return {pathway: (nes, fdr)} from GSEA prerank on the signed Wald stat.
    rnk = (df_ct[["gene", "stat"]]
           .dropna(subset=["stat"])
           .groupby("gene", as_index=False)["stat"].mean()
           .sort_values("stat", ascending=False))
    if len(rnk) < GSEA_MIN_SIZE:
        return {}
    try:
        res = gp.prerank(rnk=rnk, gene_sets=sets,
                         min_size=GSEA_MIN_SIZE, max_size=GSEA_MAX_SIZE,
                         permutation_num=GSEA_PERM, seed=GSEA_SEED,
                         threads=4, outdir=None, no_plot=True)
    except Exception:
        return {}
    return {row["Term"]: (float(row["NES"]), float(row["FDR q-val"]))
            for _, row in res.res2d.iterrows()}

hyp_by_root = defaultdict(list)
for h in HYPOTHESES:
    hyp_by_root[h["root"]].append(h)

candidates = []
for root, hyps in hyp_by_root.items():
    df = de[root]
    sets = {h["pathway"]: gene_sets[h["pathway"]] for h in hyps}
    fg, bg = df["fg"].iloc[0], df["bg"].iloc[0]
    for l3, df_ct in df.groupby("AIFI_L3"):
        scores = prerank_celltype(df_ct, sets)
        for h in hyps:
            pw = h["pathway"]
            if pw not in scores:
                continue
            nes, fdr = scores[pw]
            node = pb_node_for(l3, pw)
            if node is not None and fdr < FDR_THRESHOLD and np.sign(nes) == h["expected_nes_sign"]:
                candidates.append({
                    "from": ROOT_NODES[root], "to": node, "constraint": "whitelist",
                    "contrast": f"{fg} vs {bg}", "pathway": pw, "celltype_L3": l3,
                    "nes": round(nes, 3), "fdr": float(f"{fdr:.3e}"),
                    "expected_direction": "up" if h["expected_nes_sign"] > 0 else "down",
                    "gene_set_size": len(gene_sets[pw]), "citation": h["citation"],
                })
print(f"whitelist candidates: {len(candidates)}")

whitelist candidates: 35


In [11]:
wl_cols = ["from", "to", "constraint", "contrast", "pathway", "celltype_L3",
           "nes", "fdr", "expected_direction", "gene_set_size", "citation"]
whitelist = pd.DataFrame(candidates, columns=wl_cols)
if len(whitelist):
    whitelist = whitelist.sort_values(["from", "pathway", "fdr"]).reset_index(drop=True)
whitelist.to_csv(PROCESSED / "de_whitelist_candidates.csv", index=False)
print("wrote de_whitelist_candidates.csv:", whitelist.shape)
if len(whitelist):
    print("\ncandidates per (root, pathway):")
    print(whitelist.groupby(["from", "pathway"]).size().to_string())
whitelist.head(20)

wrote de_whitelist_candidates.csv: (35, 11)

candidates per (root, pathway):
from                             pathway  
cmv.igg_serology_interpretation  ifn_gamma    30
subject.ageGroup                 tnfa_nfkb     2
subject.biologicalSex            ifn_alpha     3


,from,to,constraint,contrast,pathway,celltype_L3,nes,fdr,expected_direction,gene_set_size,citation
0,cmv.igg_serology_interpretation,pb.cd56bright_nk_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,CD56bright NK cell,2.023,0.000000,up,200,Sylwester 2005 PMID:16147978
1,cmv.igg_serology_interpretation,pb.cm_cd8_t_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,CM CD8 T cell,1.862,0.000000,up,200,Sylwester 2005 PMID:16147978
2,cmv.igg_serology_interpretation,pb.core_cd14_monocyte_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,Core CD14 monocyte,2.093,0.000000,up,200,Sylwester 2005 PMID:16147978
3,cmv.igg_serology_interpretation,pb.core_naive_cd4_t_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,Core naive CD4 T cell,1.654,0.000000,up,200,Sylwester 2005 PMID:16147978
4,cmv.igg_serology_interpretation,pb.gzmb_cd27_em_cd4_t_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,GZMB- CD27+ EM CD4 T cell,1.990,0.000000,up,200,Sylwester 2005 PMID:16147978
5,cmv.igg_serology_interpretation,pb.gzmb_cd27_em_cd4_t_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,GZMB- CD27- EM CD4 T cell,1.636,0.000000,up,200,Sylwester 2005 PMID:16147978
6,cmv.igg_serology_interpretation,pb.gzmk_cd56dim_nk_cell_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,GZMK+ CD56dim NK cell,1.870,0.000000,up,200,Sylwester 2005 PMID:16147978
7,cmv.igg_serology_interpretation,pb.gzmk_memory_cd4_treg_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,GZMK+ memory CD4 Treg,1.787,0.000000,up,200,Sylwester 2005 PMID:16147978
8,cmv.igg_serology_interpretation,pb.isg_cd16_monocyte_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,ISG+ CD16 monocyte,1.695,0.000000,up,200,Sylwester 2005 PMID:16147978
9,cmv.igg_serology_interpretation,pb.isg_mait_ifn_gamma,whitelist,Positive vs Negative,ifn_gamma,ISG+ MAIT,2.552,0.000000,up,200,Sylwester 2005 PMID:16147978


## Validation & summary

In [12]:
# Re-load the three outputs and assert the contract invariants.
bl = pd.read_csv(PROCESSED / "bn_constraints.csv")
tr = pd.read_csv(PROCESSED / "bn_node_tiers.csv")
wl = pd.read_csv(PROCESSED / "de_whitelist_candidates.csv")

assert (bl["constraint"] == "blacklist").all(), "bn_constraints must be all blacklist"
assert set(bl["to"]) <= set(ROOT_COLS), "blacklist edges must point INTO roots only"
assert set(tr[tr.tier == 0]["node"]) == set(ROOT_COLS), "tier 0 must be exactly the roots"
if len(wl):
    assert (wl["constraint"] == "whitelist").all()
    assert wl["to"].str.startswith("pb.").all(), "whitelist targets pb.* only"
    assert wl["from"].isin(ROOT_COLS).all(), "whitelist edges must originate from a root"

print("OK — all invariants hold.")
print(f"\nbn_constraints.csv         : {len(bl):>4} blacklist edges into {bl['to'].nunique()} roots")
print(f"bn_node_tiers.csv          : {len(tr):>4} nodes ({(tr.tier==0).sum()} roots + {(tr.tier==1).sum()} tier-1)")
print(f"de_whitelist_candidates.csv: {len(wl):>4} whitelist candidates (NOT auto-loaded)")

OK — all invariants hold.

bn_constraints.csv         : 1740 blacklist edges into 3 roots
bn_node_tiers.csv          :  581 nodes (3 roots + 578 tier-1)
de_whitelist_candidates.csv:   35 whitelist candidates (NOT auto-loaded)



### Definition of done

- [x] DE treated as **constraints**, never nodes; not a `*_baseline_wide.csv` block.
- [x] **Primary `bn_constraints.csv`** = structural blacklist (Rules 1 & 2), fed to `hc()`.
- [x] **`bn_node_tiers.csv`** = regenerable source of truth for the R side.
- [x] **`de_whitelist_candidates.csv`** = back-pocket biology-first whitelist (DE-localized,
      literature-justified, **not** wired into the model).
- [x] Flu-vaccine contrast parked with documented rationale.
- [x] Whitelist targets `pb.*` only; Hallmark sets reused verbatim from Person B.

### Caveats carried forward
- **Regenerate `bn_constraints.csv`** against the final joined column list once Person A's
  clinical spine exists (materialized form errors in `hc()` if it names an absent node).
- **CMV root name** (`cmv.igg_serology_interpretation`) is a placeholder — reconcile with Person A.
- Whitelist candidates are **not** consensus edges; promotion is a team decision, edge-by-edge.